Diseñar una IA que entienda a los jugadores

Título: "Phantom Arena: Entrenando una IA para clasificar, predecir y agrupar jugadores"

Descripción:

En este ejercicio, deberás entrenar un modelo de Machine Learning para predecir el estilo de juego de un jugador, el número de victorias esperadas y asignar a cada jugador un grupo de características. Para ello, deberás crear una clase llamada GameModel que gestione y entrene los modelos correspondientes.

En lugar de cargar datos desde un archivo CSV, recibirás un conjunto de datos de prueba directamente en el código. Usarás estos datos para entrenar y evaluar tus modelos.

Tareas:

Crear la clase Player: Esta clase representará a un jugador y debe contener los siguientes atributos:

player_name: nombre del jugador (string).

character_type: tipo de personaje (string). Puede ser "mage", "tank", "archer", "assassin".

avg_session_time: tiempo promedio por sesión en minutos (float).

matches_played: número total de partidas jugadas (int).

aggressive_actions: cantidad de acciones agresivas realizadas (int).

defensive_actions: cantidad de acciones defensivas realizadas (int).

items_bought: cantidad de objetos comprados (int).

victories: número de victorias (int).

style: estilo de juego del jugador ("aggressive" o "strategic", sólo uno de estos).

Crear la clase GameModel: Esta clase debe ser capaz de:

Recibir una lista de jugadores y almacenarlos.

Entrenar tres modelos diferentes:

Modelo de clasificación: para predecir el estilo de juego del jugador (aggressive o strategic).

Modelo de regresión: para predecir el número de victorias de un jugador.

Modelo de clustering: para asignar a cada jugador un grupo basado en sus características.

Proveer métodos para:

Predecir el estilo de juego de un jugador.

Predecir las victorias de un jugador.

Asignar un cluster a un jugador.

Pruebas:

No deberás usar ningún archivo externo. Todos los datos serán proporcionados directamente en el código, en forma de una lista de objetos Player.

Después de entrenar los modelos, deberás hacer predicciones para un jugador de prueba.

🧩 Datos de prueba

Se te proporcionará un conjunto de datos de prueba como el siguiente:

players_data = [
    
    Player("P1", "mage", 40, 30, 90, 50, 20, 18, "aggressive"),
    
    Player("P2", "tank", 60, 45, 50, 120, 25, 24, "strategic"),
    
    Player("P3", "archer", 50, 35, 95, 60, 22, 20, "aggressive"),
    
    Player("P4", "tank", 55, 40, 60, 100, 28, 22, "strategic"),
]


🧩 Ejemplo de uso

# Crear datos de prueba para varios jugadores
players_data = [
    
    Player("P1", "mage", 40, 30, 90, 50, 20, 18, "aggressive"),
    
    Player("P2", "tank", 60, 45, 50, 120, 25, 24, "strategic"),
    
    Player("P3", "archer", 50, 35, 95, 60, 22, 20, "aggressive"),
    
    Player("P4", "tank", 55, 40, 60, 100, 28, 22, "strategic"),
]
 
# Instanciar el modelo con los datos de los jugadores

model = GameModel(players_data)
 
# Entrenar los modelos

model.train_classification_model()

model.train_regression_model()

model.train_clustering_model()
 
# Crear un nuevo jugador para realizar predicciones

new_player = Player("TestPlayer", "mage", 42, 33, 88, 45, 21, 0)
 
# Realizar predicciones

predicted_style = model.predict_style(new_player)

predicted_victories = model.predict_victories(new_player)

predicted_cluster = model.assign_cluster(new_player)
 
# Imprimir los resultados de las predicciones

print(f"Estilo de juego predicho para {new_player.player_name}: {predicted_style}")

print(f"Victorias predichas para {new_player.player_name}: {predicted_victories:.2f}")

print(f"Cluster asignado a {new_player.player_name}: {predicted_cluster}")


🧩 Salida esperada

Estilo de juego predicho para TestPlayer: aggressive
Victorias predichas para TestPlayer: 17.70
Cluster asignado a TestPlayer: 0


🧩 Tarea Opcional: Mostrar jugadores por cluster

Para este ejercicio adicional, te proponemos una función que te permitirá visualizar los jugadores agrupados por clusters. Esta función será útil para explorar y entender cómo el modelo de clustering (KMeans) ha agrupado a los jugadores en función de sus características.

Descripción de la tarea:

Debes implementar una función que imprima los jugadores asignados a cada cluster después de que el modelo KMeans haya sido entrenado. Cada cluster debe ser visualizado con el nombre del jugador, el tipo de personaje y su estilo de juego.

Especificaciones:

Utiliza el modelo KMeans entrenado en el ejercicio anterior.

La función debe recorrer los diferentes clusters y mostrar los jugadores pertenecientes a cada uno, con los siguientes detalles:

Nombre del jugador

Tipo de personaje

Estilo de juego

La salida debe ser algo como:

Cluster 0:

P1 - Mage - Aggressive

P3 - Archer - Aggressive

Cluster 1:

P2 - Tank - Strategic

P4 - Tank - Strategic

Consejos:

Puedes utilizar model.cluster_model.labels_ para obtener las asignaciones de los clusters.

Convierte los datos de los jugadores en un DataFrame para facilitar la manipulación y visualización de la información.

La función debe imprimir los jugadores por cada cluster, y para ello puedes agrupar los jugadores según el valor de model.cluster_model.labels_.



Practicar con modelos de clasificación, regresión y clustering


# Solucion:

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

# --- CLASE 1: EL JUGADOR ---
class Player:
    def __init__(self, player_name, character_type, avg_session_time, matches_played, 
                 aggressive_actions, defensive_actions, items_bought, victories, style=None):
        self.player_name = player_name
        self.character_type = character_type
        self.avg_session_time = avg_session_time
        self.matches_played = matches_played
        self.aggressive_actions = aggressive_actions
        self.defensive_actions = defensive_actions
        self.items_bought = items_bought
        self.victories = victories
        self.style = style  # Puede ser None si es un jugador nuevo a predecir

    # Método auxiliar para convertir el objeto a un diccionario (útil para Pandas)
    def to_dict(self):
        return {
            'player_name': self.player_name,
            'character_type': self.character_type,
            'avg_session_time': self.avg_session_time,
            'matches_played': self.matches_played,
            'aggressive_actions': self.aggressive_actions,
            'defensive_actions': self.defensive_actions,
            'items_bought': self.items_bought,
            'victories': self.victories,
            'style': self.style
        }

# --- CLASE 2: EL CEREBRO DE LA IA ---
class GameModel:
    def __init__(self, players):
        self.players = players
        # Convertimos la lista de objetos Player a DataFrame
        self.df = pd.DataFrame([p.to_dict() for p in players])
        
        # PREPROCESAMIENTO IMPORTANTE:
        # Las IAs no entienden "mage" o "tank", solo números.
        # Usamos un mapeo fijo para asegurar consistencia entre entrenamiento y predicción.
        self.char_map = {'mage': 1, 'tank': 2, 'archer': 3, 'assassin': 4}
        
        # Aplicamos el mapeo
        self.df['char_code'] = self.df['character_type'].map(self.char_map).fillna(0)
        
        # Definimos las características (X) que usaremos para aprender
        self.features = ['char_code', 'avg_session_time', 'matches_played', 
                         'aggressive_actions', 'defensive_actions', 'items_bought']
        
        # Inicializamos los modelos como None
        self.clf_model = None # Clasificación
        self.reg_model = None # Regresión
        self.cluster_model = None # Clustering

    def train_classification_model(self):
        # Objetivo: Predecir 'style' (aggressive/strategic)
        X = self.df[self.features]
        y = self.df['style']
        
        self.clf_model = LogisticRegression(random_state=42)
        self.clf_model.fit(X, y)

    def train_regression_model(self):
        # Objetivo: Predecir 'victories' (número)
        X = self.df[self.features]
        y = self.df['victories']
        
        self.reg_model = LinearRegression()
        self.reg_model.fit(X, y)

    def train_clustering_model(self):
        # Objetivo: Agrupar por similitud (sin etiquetas)
        X = self.df[self.features]
        
        # Usamos n_clusters=2 porque tenemos pocos datos en el ejemplo,
        # en un juego real usaríamos "El método del codo" para decidir cuántos.
        self.cluster_model = KMeans(n_clusters=2, random_state=42, n_init=10)
        self.cluster_model.fit(X)
        
        # Guardamos las etiquetas en el DF para la tarea opcional
        self.df['cluster_label'] = self.cluster_model.labels_

    # --- MÉTODOS DE PREDICCIÓN ---
    
    def _prepare_single_player(self, player):
        """Convierte un objeto Player individual al formato numérico que espera la IA"""
        # Crear diccionario y mapear el tipo de personaje
        p_data = player.to_dict()
        p_data['char_code'] = self.char_map.get(player.character_type, 0)
        
        # Convertir a DataFrame de 1 fila con las columnas correctas
        df_single = pd.DataFrame([p_data])
        return df_single[self.features]

    def predict_style(self, player):
        X_new = self._prepare_single_player(player)
        return self.clf_model.predict(X_new)[0]

    def predict_victories(self, player):
        X_new = self._prepare_single_player(player)
        return self.clf_model.predict(X_new)[0] if False else self.reg_model.predict(X_new)[0]

    def assign_cluster(self, player):
        X_new = self._prepare_single_player(player)
        return self.cluster_model.predict(X_new)[0]
    
    # --- TAREA OPCIONAL ---
    def print_players_by_cluster(self):
        print("\n--- Jugadores agrupados por Cluster ---")
        # Agrupar el DataFrame por la etiqueta del cluster
        for cluster_id, group in self.df.groupby('cluster_label'):
            print(f"Cluster {cluster_id}:")
            for _, row in group.iterrows():
                print(f"  - {row['player_name']} ({row['character_type']}) - {row['style']}")
            print("")

# --- BLOQUE DE EJECUCIÓN (MAIN) ---
if __name__ == "__main__":
    # 1. Datos de entrenamiento (Mínimo viable)
    players_data = [
        Player("P1", "mage", 40, 30, 90, 50, 20, 18, "aggressive"),
        Player("P2", "tank", 60, 45, 50, 120, 25, 24, "strategic"),
        Player("P3", "archer", 50, 35, 95, 60, 22, 20, "aggressive"),
        Player("P4", "tank", 55, 40, 60, 100, 28, 22, "strategic"),
        # Añadimos un par más para dar robustez al ML
        Player("P5", "assassin", 35, 25, 100, 30, 25, 15, "aggressive"), 
        Player("P6", "mage", 65, 50, 45, 110, 30, 30, "strategic")
    ]

    # 2. Instanciar el GameModel
    print("Inicializando Phantom Arena AI...")
    model = GameModel(players_data)

    # 3. Entrenar los modelos
    print("Entrenando modelos...")
    model.train_classification_model()
    model.train_regression_model()
    model.train_clustering_model()

    # 4. Crear un nuevo jugador (El "TestPlayer")
    # Nota: victories=0 y style=None porque es lo que queremos averiguar
    new_player = Player("TestPlayer", "mage", 42, 33, 88, 45, 21, 0)

    # 5. Realizar predicciones
    predicted_style = model.predict_style(new_player)
    predicted_victories = model.predict_victories(new_player)
    predicted_cluster = model.assign_cluster(new_player)

    # 6. Mostrar resultados
    print(f"\nResultados para {new_player.player_name}:")
    print(f"Estilo de juego predicho: {predicted_style}")
    print(f"Victorias esperadas: {predicted_victories:.2f}")
    print(f"Cluster asignado: {predicted_cluster}")
    
    # 7. Ejecutar tarea opcional
    model.print_players_by_cluster()

Inicializando Phantom Arena AI...
Entrenando modelos...

Resultados para TestPlayer:
Estilo de juego predicho: aggressive
Victorias esperadas: 20.93
Cluster asignado: 0

--- Jugadores agrupados por Cluster ---
Cluster 0:
  - P1 (mage) - aggressive
  - P3 (archer) - aggressive
  - P5 (assassin) - aggressive

Cluster 1:
  - P2 (tank) - strategic
  - P4 (tank) - strategic
  - P6 (mage) - strategic



C:\Users\Vaquita\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Casos de prueba

Suspenso: 0, Aprobado: 3 de 3 pruebas

test_assign_cluster

test_predict_style

test_predict_victories